In [ ]:
# build using langchain to 

# !pip install -q langgraph langchain-core langchain-openai python-dotenv langchain-community pypdf

print("All dependencies installed!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
anaconda-cloud-auth 0.1.3 requires pydantic<2.0, but you have pydantic 2.12.4 which is incompatible.
python-lsp-black 1.2.1 requires black>=22.3.0, but you have black 0.0 which is incompatible.
✅ All dependencies installed!


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv()
if env_path:
    load_dotenv(env_path)
    print("Loaded configuration from .env file")
else:
    print("No .env file found - using environment variables or hardcoded keys")


# Import official packages
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# Verify API keys
print("\n🔑 API Key Status:")
if os.getenv('OPENAI_API_KEY') and os.getenv('OPENAI_API_KEY'):
    print("OPENAI_API_KEY  loaded")

print("\n✅ All imports successful!")

Loaded configuration from .env file


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



🔑 API Key Status:
OPENAI_API_KEY  loaded

✅ All imports successful!


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Only keep post title, headers, and content from the full HTML.
loader = PyPDFLoader('../data/BMW/BMW_Annual_Report_2021.pdf')
docs = loader.load()

print(f"Total characters for doc 1: {len(docs[0].page_content)}")


text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs)

len(doc_splits)

Total characters for doc 1: 288


742

# retrieve

In [3]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

In [7]:
from langchain_classic.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "retrieve_reports",
    "Retrieve info on automobiles",
)

In [8]:
retriever_tool.invoke({"query": "What was BMW's total revenue in 2021?"})


'T otal\u200a3 2,465,021 2,486,149 2,537,504 2,325,179 2,521,514 8.4\nProduction by brand     \xa0 \xa0\nBMW \u200a4 2,123,947 2,168,496 2,205,841 1,980,740 2,166,644 9.4\n\nBMW  Group 16,289 5,464 8,938 8,061 7,351 – 2,597\nin % 2021 2020\nAutomotive 12.0 12.0\nMotorcycles 12.0 12.0\nFinancial Services 13.4 13.4\nValue added  \nGroup\nearnings amount –  \ncost of capital\nearnings amount –  \n(cost of capital rate x\ncapital employed)\n=\n=\n47\n\nenvironment more volatile and calling for even greater flexibility from company and \nworkforce alike. However, the prudent leadership of the Board of Management and \nthe tremendous hard work of our employees helped make 2021 a highly successful \nfinancial year for the BMW Group. With a new record of over 2.5 million BMW brand \nvehicles delivered, we are now the leading manufacturer in the premium segment \nworldwide. With great resolve, the Board of Management continued to develop the\n\nin %\n\xa0 2021 2020\xa0 2021 2020\xa0 2021 2020\x

In [10]:
from langgraph.graph import MessagesState
from langchain_openai import ChatOpenAI

# Create model
response_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
def generate_query_or_respond(state: MessagesState):
    """Call the model to generate a response based on the current state."""
    response = (
        response_model
        .bind_tools([retriever_tool])
        .invoke(state["messages"])
    )
    return {"messages": [response]}


In [ ]:
input = {"messages": [{"role": "user", "content": " "}]}
generate_query_or_respond(input)["messages"][-1].pretty_print()

================================== Ai Message ==================================

BMW's total revenue in 2021 was approximately €111.2 billion.
